# Chapter 2 - working with text data

## Goal: Preparing dataset for the LLM

In [64]:
import os
import urllib.request

In [65]:
## raw data fetch was being done as html type -- so better not to do it
# if not os.path.exists('the-verdict.txt'):
#     url = ("https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/refs/heads/main/ch02/01_main-chapter-code/the-verdict.txt")
#     file_path = 'the-verdict.txt'
#     urllib.request.urlretrieve(url, file_path)
    

In [66]:
with open("the-verdict.txt", 'r', encoding='utf-8') as f:
    raw_text = f.read()

In [67]:
len(raw_text)

20479

In [68]:
import re

text = "Hello, world. This, is a test."
result = re.split(r'([,.]|\s)', text)

In [70]:
print(result)

['Hello', ',', '', ' ', 'world', '.', '', ' ', 'This', ',', '', ' ', 'is', ' ', 'a', ' ', 'test', '.', '']


In [71]:
result = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
result = [item.strip() for item in result if item.strip()]
preprocessed = result

In [72]:
len(preprocessed)

4690

In [73]:
len(set(preprocessed))

1130

In [182]:
# raw_text

## converting tokens into tokens IDS

In [75]:
"""
building vocabulary
building training data in to tokens (converting tokens of training data to tokens)
"""

'\nbuilding vocabulary\nbuilding training data in to tokens (converting tokens of training data to tokens)\n'

In [76]:
all_words = sorted(set(preprocessed))

In [25]:
vocab_size = len(all_words)

In [26]:
vocab = {token: integer for integer, token in enumerate(all_words)}

In [27]:
len(vocab)

1130

In [29]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s, i in vocab.items()}
        
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]

        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

In [30]:
tokenizer = SimpleTokenizerV1(vocab)

In [35]:
text = """"It's the last time he painted, you know,"
Mrs. Gisburn said with pardonable pride.
"""

In [36]:
ids = tokenizer.encode(text)

In [39]:
print(ids)

[1, 56, 2, 850, 988, 602, 1011, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


In [38]:
tokenizer.decode(ids)

'" It\' s the last time he painted, you know," Mrs. Gisburn said with pardonable pride.'

## Adding special context tokens
### addding tokens for unknown words and end of text

In [100]:
text = 'Hello, do you like tea. is this-- a test?'

In [43]:
# tokenizer.encode(text) -- `hello` not present in the vocaculary

In [44]:
'''
there are algorithms which handles the unknown words by splitting them down in the known words -- how do they work for chinese or new languagge character never seen (the basic building entity itself is never seen before)
'''

'\nthere are algorithms which handles the unknown words by splitting them down in the known words -- how do they work for chinese or new languagge character never seen (the basic building entity itself is never seen before)\n'

In [79]:
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["|<endoftext>|", "<|unk|>"])

In [80]:
len(all_tokens)

1132

In [77]:
all_words = sorted(set(preprocessed))

In [78]:
vocab_size = len(all_words)

In [81]:
vocab = {token: integer for integer, token in enumerate(all_tokens)}

In [82]:
len(vocab)

1132

In [101]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s, i in vocab.items()}
        
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]

        #adding special context tokens
        preprocessed = [item if item in self.str_to_int else "<|unk|>" for item in preprocessed]

        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

In [102]:
len(vocab)

1132

In [103]:
tokenizer = SimpleTokenizerV2(vocab)

In [104]:
tokenizer.encode(text)

[1131, 5, 355, 1126, 628, 975, 7, 1131, 999, 6, 115, 1131, 10]

In [105]:
tokenizer.decode(tokenizer.encode(text))

'<|unk|>, do you like tea. <|unk|> this -- a <|unk|>?'

In [109]:
### importing tiktoken tokenizer (same as the one used in gpt2, 3 and 4)

In [110]:
import tiktoken

## Data Sampling with Sliding window

In [111]:
import torch
from torch.utils.data import Dataset, DataLoader

In [125]:
class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(txt)

        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i+1: i+1+max_length]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def  __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [126]:
def create_dataloader_v1(txt, batch_size = 4, max_length = 256, stride = 128,
                         shuffle = True, drop_last = True, num_workers = 0):
    tokenizer = tiktoken.get_encoding('gpt2')
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    dataloader = DataLoader(dataset, batch_size = batch_size, shuffle = shuffle,
                    drop_last = drop_last, num_workers = num_workers)

    return dataloader

In [127]:
with open('the-verdict.txt', 'r', encoding = 'utf-8') as f:
    raw_text = f.read()

In [132]:
dataloader = create_dataloader_v1(raw_text,  batch_size = 1,
                                 max_length = 4, stride = 4,
                                 shuffle = False)
data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


In [133]:
print(next(data_iter))

[tensor([[1807, 3619,  402,  271]]), tensor([[ 3619,   402,   271, 10899]])]


In [134]:
for i in range(10):
    print(next(data_iter))

[tensor([[10899,  2138,   257,  7026]]), tensor([[ 2138,   257,  7026, 15632]])]
[tensor([[15632,   438,  2016,   257]]), tensor([[ 438, 2016,  257,  922]])]
[tensor([[ 922, 5891, 1576,  438]]), tensor([[5891, 1576,  438,  568]])]
[tensor([[568, 340, 373, 645]]), tensor([[ 340,  373,  645, 1049]])]
[tensor([[1049, 5975,  284,  502]]), tensor([[5975,  284,  502,  284]])]
[tensor([[ 284, 3285,  326,   11]]), tensor([[3285,  326,   11,  287]])]
[tensor([[ 287,  262, 6001,  286]]), tensor([[ 262, 6001,  286,  465]])]
[tensor([[  465, 13476,    11,   339]]), tensor([[13476,    11,   339,   550]])]
[tensor([[  550,  5710,   465, 12036]]), tensor([[ 5710,   465, 12036,    11]])]
[tensor([[  11, 6405,  257, 5527]]), tensor([[ 6405,   257,  5527, 27075]])]


In [137]:
###increasing batch size and other stuff
dataloader = create_dataloader_v1(raw_text,  batch_size = 8,
                                 max_length = 4, stride = 4,
                                 shuffle = False)
data_iter = iter(dataloader)
inputs, target = next(data_iter)
print(f"Inputs: {inputs}")
print(f"Targets: {target}")

Inputs: tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])
Targets: tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


## Creating token embeddings

In [139]:
input_ids = torch.tensor([2, 3, 5, 1])

In [140]:
vocab_size = 6
output_dim = 3

In [141]:
torch.manual_seed(123)

In [142]:
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
print(embedding_layer.weight)

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)


In [143]:
print(embedding_layer(torch.tensor([3])))

tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)


In [144]:
print(embedding_layer(input_ids))

tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)


In [171]:
### creating based on original vocab size
tokenizer = tiktoken.get_encoding('gpt2')
vocab_size = tokenizer.n_vocab
output_dim = 256

In [172]:
token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

In [173]:
token_embedding_layer.weight.shape

torch.Size([50257, 256])

## Encoding word positions - adding positional information

In [174]:
max_length = 4
dataloader = create_dataloader_v1(
    raw_text, batch_size = 8, max_length = max_length, stride = max_length, shuffle = False
)

data_iter = iter(dataloader)
inputs, target = next(data_iter)

In [175]:
token_embeddings = token_embedding_layer(inputs)
print(token_embeddings.shape)

torch.Size([8, 4, 256])


In [176]:
### these embeddings will be trained later in chapter - 5

In [177]:
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)
pos_embeddings = pos_embedding_layer(torch.arange(context_length))
print(pos_embeddings.shape)

torch.Size([4, 256])


In [178]:
pos_embeddings

tensor([[ 0.7670,  0.8220, -0.6578,  ...,  1.9907,  0.2389, -0.4300],
        [-0.9296,  0.6107, -1.3438,  ..., -0.5825, -0.4339, -0.5746],
        [ 0.7668,  0.4623, -0.5948,  ...,  1.2851, -0.3150,  0.6180],
        [ 1.0865,  0.0889, -0.6787,  ..., -0.8388,  1.0177,  0.9643]],
       grad_fn=<EmbeddingBackward0>)

In [179]:
input_embeddings = token_embeddings + pos_embeddings

In [180]:
print(input_embeddings.shape)

torch.Size([8, 4, 256])


In [181]:
### these position embeddings starts as random values, but are trained later